In [2]:
import sys
sys.path.append('..')

import numpy as np
import mindspore
import mindspore.dataset as ds
from mindspore import Tensor
from d2l import mindspore as d2l

true_w = np.array([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

In [3]:
class SyntheticData():
    def __init__(self):
        self.features, self.labels = d2l.synthetic_data(true_w, true_b, 1000)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]
    
    def __len__(self):
        return len(self.labels)

def load_array(data_arrays, column_names, batch_size, is_train=True):  
    """构造一个MindSpore数据迭代器。"""
    dataset = ds.GeneratorDataset(data_arrays, column_names, shuffle=is_train)
    dataset = dataset.batch(batch_size)
    return dataset

batch_size = 10
data_iter = SyntheticData()
dataset = load_array(data_iter, ['features', 'labels'], batch_size)

In [4]:
next(iter(dataset))

[Tensor(shape=[10, 2], dtype=Float32, value=
 [[-0.22888152,  0.9795823 ],
  [-0.6006927 ,  2.0978146 ],
  [-0.3551168 ,  0.17106538],
  [ 0.225816  ,  0.89234185],
  [ 0.7022703 ,  0.37690485],
  [ 0.09781982,  0.58985835],
  [-1.6895691 , -0.71199685],
  [-0.9362989 ,  0.08117686],
  [-1.1256968 , -0.26578853],
  [-0.10324197, -0.25843388]]),
 Tensor(shape=[10, 1], dtype=Float32, value=
 [[ 0.39374894],
  [-4.1332893 ],
  [ 2.9085197 ],
  [ 1.6274605 ],
  [ 4.334508  ],
  [ 2.3881655 ],
  [ 3.2570887 ],
  [ 2.0458496 ],
  [ 2.8606327 ],
  [ 4.8533044 ]])]

In [5]:
# nn是神经网络的缩写
from mindspore import nn
from mindspore.common.initializer import initializer, Normal

net = nn.SequentialCell([nn.Dense(2, 1)])

In [6]:
net[0].weight = initializer(Normal(), net[0].weight.shape, mindspore.float32)
net[0].bias = initializer('zero', net[0].bias.shape, mindspore.float32)

In [7]:
loss = nn.MSELoss()

In [8]:
optimizer = nn.SGD(net.trainable_params(), learning_rate=0.03)

In [9]:
# 构造前向网络
def forward_fn(x, y):
    y_hat = net(x)
    l = loss(y_hat, y)
    return l
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in dataset:
        grad_fn = mindspore.value_and_grad(forward_fn, grad_position=None, weights=optimizer.parameters)
        l, grads = grad_fn(X, y)
        optimizer(grads)
    l = forward_fn(mindspore.Tensor(data_iter.features), mindspore.Tensor(data_iter.labels))
    print(f'epoch {epoch + 1}, loss {l.asnumpy():f}')

..epoch 1, loss 0.000389
epoch 2, loss 0.000107
epoch 3, loss 0.000107


In [10]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： [-3.00407410e-05 -2.89583206e-04]
b的估计误差： [0.00088167]
